# Kilifi pfcrt K76T analysis

**Project:** Temporal Genomic Analysis of the pfcrt K76T Chloroquine-Resistance Marker in *Plasmodium falciparum* from Kilifi, Kenya, 1994–2020.

**Question:** How did detected 76T carriage change in archived Kilifi samples, and how sensitive is the summary to mixed K/T calls, annual sampling and contributing study?

Data: [MalariaGEN Pf8](https://www.malariagen.net/data_package/open-dataset-plasmodium-falciparum-v80/) ([paper](https://doi.org/10.12688/wellcomeopenres.24031.1)).

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import json

root = Path.cwd()
if not (root / 'data/Pf8_samples.txt').exists():
    raise FileNotFoundError('Run this notebook from the project folder (data/ must be next to it).')

sys.path.insert(0, str(root))
import analysis_helpers as ah
print(ah.checks())

## 2. Rebuild the cohort from the original files

The original files are preserved. The helper checks their hashes, joins unique sample IDs, filters Kenya/Kilifi and 1994–2020, applies the provider's QC flag, and checks recorded same-case groups.

Expected snapshot: **2,078** Kilifi metadata records; **206** excluded by source QC; **1,872** retained.

In [ ]:
import pandas as pd
import numpy as np
import scipy
import matplotlib

print({'python': sys.version.split()[0], 'pandas': pd.__version__,
       'numpy': np.__version__, 'scipy': scipy.__version__,
       'matplotlib': matplotlib.__version__})
data, validation = ah.build_dataset(root)
print(json.dumps(validation, indent=2))
print(data[['sample_id','year','study','raw_crt76_call','marker_call']].head(8).to_string(index=False))

**Group task:** Choose one included and one excluded record from the inclusion audit. Explain their decisions using the source fields. Check that the sample IDs stay as text and that recorded case-group linkage is not treated as a new independent sample identifier.

## 3. Understand K-only, T-only and mixed calls

K means lysine and T means threonine at PfCRT position 76. `K,T` and `T,K` are both mixed K/T. The primary outcome counts a mixed sample as having detected 76T. The secondary outcome excludes mixed samples from its denominator.

- Primary: `(T-only + mixed) / (K-only + T-only + mixed)`.
- Secondary: `T-only / (K-only + T-only)`.

Neither measure estimates the proportion of all parasite clones carrying 76T. A K-only call at one codon does not prove clinical chloroquine sensitivity.

In [ ]:
print(data['marker_call'].value_counts().to_string())
print('\nRaw mixed-call encodings:')
print(data.loc[data['marker_call'].eq('mixed_KT'), 'raw_crt76_call'].value_counts().to_string())
print('\nOutcome encoding:')
print(data[['marker_call','has_76T','unambiguous_76T']].drop_duplicates().to_string(index=False))

callable_n = int(data['marker_callable'].sum())
carrier_n = int(data['has_76T'].sum())
unambiguous_n = int(data['unambiguous_76T'].notna().sum())
t_only_n = int(data['unambiguous_76T'].sum())
print(f'Across all retained years, primary = {carrier_n}/{callable_n} = {carrier_n/callable_n:.3f}')
print(f'Across all retained years, secondary = {t_only_n}/{unambiguous_n} = {t_only_n/unambiguous_n:.3f}')
print('These pooled values combine very different years; the temporal summaries answer the project question.')

## 4. Summarize annual coverage and prevalence

Each outcome has a 95% Wilson interval. Proportions use a 0–1 scale. Years with no retained samples have zero counts and missing proportions. The intervals do not remove archive-selection bias.

In [ ]:
annual = ah.summarize(data)
columns = ['year','n_qc','n_K_only','n_T_only','n_mixed_KT',
           'carrier_proportion','carrier_ci_low','carrier_ci_high',
           'unambiguous_proportion']
print(annual[columns].round(4).to_string(index=False))
missing_years = annual.loc[annual['n_qc'].eq(0), 'year'].tolist()
print('No samples in:', missing_years)
assert annual.loc[annual['n_qc'].eq(0), 'carrier_proportion'].isna().all()

**Group task:** Select one observed year and calculate both proportions manually from its category counts. Then explain why 0 detected carriers in an observed year differs from a year with 0 samples.

## 5. Compare predefined periods and sampling choices

The five period groups are fixed in the starter methods. They are not estimated change points or official policy periods. Pooling weights heavily sampled years more strongly; an equally weighted mean uses each observed year's estimate once. Both remain summaries of the archive.

The main-study restriction uses `1132-PF-K1000G-DBS-KE-BEJON`, the largest contributor (1,608 retained samples). It helps examine sensitivity to study composition but cannot eliminate all sampling differences.

In [ ]:
period = ah.summarize(data, 'time_period')
sensitivity, annual_main = ah.sampling_sensitivity(data)
print(period[['time_period','n_callable','carrier_proportion',
              'carrier_ci_low','carrier_ci_high','unambiguous_proportion']].round(4).to_string(index=False))
print('\nSensitivity comparisons:')
print(sensitivity.round(4).to_string(index=False))
print('\nStudy counts by observed year:')
study_by_year = pd.crosstab(data['year'], data['study']).reindex(ah.YEARS, fill_value=0)
print(study_by_year.to_string())

**Group task:** Identify the periods where the main-study restriction changes the estimate most. Quantify the difference in percentage points, and discuss whether it changes the biological interpretation. Do not call an equally weighted mean a correction to population prevalence.

## 6. Generate figures and tables

Tables and figures are written to `outputs/`.

In [ ]:
results = ah.run_analysis(root)
output_dir = results['output_dir']
print('Outputs saved in:', output_dir)

import matplotlib.pyplot as plt
for name in ['01_yearly_sample_counts.png', '02_marker_composition.png', '03_temporal_prevalence.png']:
    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.imshow(plt.imread(output_dir / name))
    ax.axis('off')
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 7. An exploratory association test

This optional comparison uses a contingency table of **predefined period × detected 76T carriage**. The code checks expected cell counts before presenting the chi-square approximation. This is an exploratory association test, not a test of a monotonic trend, a causal policy evaluation or a clinical efficacy result. Discuss the independence and sampling assumptions with your lecturer.

In [ ]:
period_test = ah.exploratory_period_test(data)
print(json.dumps(period_test, indent=2))

## 8. Write the group's interpretation

Complete these prompts in your own words, referring to the generated tables and figures:

1. **Main observation:** [Describe the temporal pattern with denominators and uncertainty.]
2. **Mixed-call sensitivity:** [Explain how and why the definitions differ.]
3. **Sampling sensitivity:** [Compare pooled/equal-year and all/main-study summaries.]
4. **Biological explanation:** [Use primary literature; distinguish a plausible explanation from a causal result.]
5. **Comparison with earlier Kilifi work:** [Read Omedo et al. and Wamae et al.; consider sample overlap.]
6. **Limits:** [Address archive sampling, missing years, study composition, single-marker scope and the historical end date.]
7. **Feedback-driven changes:** [Record lecturer comments and any revised analytical choices.]

The data do not support a treatment recommendation or a statement of current prevalence. Record actual group contributions and acknowledge starter or AI-assisted material according to your course rules.
